# 🎧 LimpiarAudio · Aislar sonidos con AudioSep (Google Colab / GPU)

Modo **GPU en la nube** para la FASE 5 cuando no tienes GPU local.
AudioSep aísla **cualquier sonido descrito en lenguaje natural** (en inglés).

**Antes de empezar:** menú `Entorno de ejecución` → `Cambiar tipo de entorno` → **GPU (T4)**.

Flujo: instalar → descargar checkpoints → cargar modelo → subir tu audio →
escribir la descripción del sonido → descargar el sonido aislado.

In [ ]:
# 1) Clonar AudioSep e instalar dependencias de inferencia
!git clone https://github.com/Audio-AGI/AudioSep.git
%cd AudioSep
!pip install -q torchlibrosa "transformers==4.30.2" lightning braceexpand webdataset ftfy h5py pandas soundfile "librosa==0.10.1"

In [ ]:
# 2) Descargar los checkpoints (AudioSep ~1.3 GB + CLAP ~2.4 GB)
!mkdir -p checkpoint
!wget -q --show-progress -O checkpoint/audiosep_base_4M_steps.ckpt \
  https://huggingface.co/spaces/Audio-AGI/AudioSep/resolve/main/checkpoint/audiosep_base_4M_steps.ckpt
!wget -q --show-progress -O checkpoint/music_speech_audioset_epoch_15_esc_89.98.pt \
  https://huggingface.co/spaces/Audio-AGI/AudioSep/resolve/main/checkpoint/music_speech_audioset_epoch_15_esc_89.98.pt

In [ ]:
# 3) Cargar el modelo (usa GPU si el entorno la tiene)
import torch
from pipeline import build_audiosep, separate_audio

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)
model = build_audiosep(
    config_yaml='config/audiosep_base.yaml',
    checkpoint_path='checkpoint/audiosep_base_4M_steps.ckpt',
    device=device,
)

In [ ]:
# 4) Sube tu audio (el mismo que usas en LimpiarAudio: source.wav o cualquier WAV/MP3)
from google.colab import files
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print('Audio:', audio_file)

In [ ]:
# 5) Aislar un sonido. Escribe la descripción EN INGLÉS (igual que hace la app:
#    usa la etiqueta original de PANNs, p. ej. 'Speech', 'Music', 'Applause',
#    'Engine', 'Dog', 'Piano', 'Drums'...)
query = 'Applause'  # <-- cámbialo por el sonido que quieras aislar
output_file = 'aislado.wav'

separate_audio(model, audio_file, query, output_file, device=device, use_chunk=True)

from IPython.display import Audio, display
print('Resultado:')
display(Audio(output_file))
files.download(output_file)

### Notas
- Con **GPU T4** una pista de decenas de segundos se aísla en pocos segundos.
- Puedes repetir la celda 5 con distintas descripciones para aislar varios sonidos.
- Descarga cada `aislado.wav`, renómbralo y súbelo/mézclalo como pista extra.
- La descripción debe ir **en inglés** (el codificador de texto CLAP se entrenó en inglés).